# 대출 승인 예측 ML 모델

고객의 대출 승인 여부를 예측하는 이진 분류 모델을 구축합니다.  
    • 데이터: training_data.csv (5,000건, 13개 변수)  
    • 모델: Logistic Regression, Random Forest, XGBoost, LightGBM 중에 2개 선택해서 비교  
    • 평가 지표: 정확도, 정밀도, 재현율, F1 Score (불균형 데이터 대응)  

## 선택한 모델
- Logistic Regression
  - 대출 승인은 금융 도메인 — 모델의 해석 가능성이 중요 (규제/감사 대응)
  - 베이스라인 역할: 복잡한 모델과 성능 비교 기준점
  - 수치형 피처만 있어 전처리 부담 없음
- LightGBM
  - 5,000행 정도의 테이블형 데이터에서 Gradient Boosting 계열이 압도적으로 강함
  - XGBoost보다 학습 속도 빠름, 메모리 효율 좋음
  - class_weight / scale_pos_weight로 클래스 불균형(71:29) 쉽게 대응
  - Random Forest보다 일반적으로 성능 우위

# 0. Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 데이터 분석

### 1-1. 데이터 탐색 및 결측치 확인

In [ ]:
df = pd.read_csv('training_data.csv')
print(df.shape)
print(df.info())
print('------')
# 클래스 불균형 확인
print(df['승인여부'].value_counts(normalize=True))
# 결측치 확인
print(df.isnull().sum())


(5000, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   나이      5000 non-null   int64
 1   연봉      5000 non-null   int64
 2   재산      5000 non-null   int64
 3   예금      5000 non-null   int64
 4   자동차배기량  5000 non-null   int64
 5   자녀수     5000 non-null   int64
 6   신청금액    5000 non-null   int64
 7   신청기간월   5000 non-null   int64
 8   학력점수    5000 non-null   int64
 9   결혼점수    5000 non-null   int64
 10  회사규모점수  5000 non-null   int64
 11  대출분류점수  5000 non-null   int64
 12  승인여부    5000 non-null   int64
dtypes: int64(13)
memory usage: 507.9 KB
None
------
승인여부
1    0.7118
0    0.2882
Name: proportion, dtype: float64
나이        0
연봉        0
재산        0
예금        0
자동차배기량    0
자녀수       0
신청금액      0
신청기간월     0
학력점수      0
결혼점수      0
회사규모점수    0
대출분류점수    0
승인여부      0
dtype: int64


&rarr;  클래스 불균형이 심하므로 불균형 해소 필요

**클래스 불균형(Class Imbalance)**이란 분류 문제에서 각 클래스(결과값)에 속하는 **데이터의 양이 현저하게 차이 나는 상태**를 말합니다.

대출 승인 예측을 예로 들면, 전체 5,000건의 데이터 중 '승인(1)'은 4,500건인데 '거절(0)'은 500건밖에 없는 경우입니다. 이때 '승인'은 **다수 클래스(Majority Class)**, '거절'은 **소수 클래스(Minority Class)**가 됩니다.

---

## 1. 클래스 불균형이 왜 문제일까?

모델은 기본적으로 '정확도(Accuracy)'를 높이는 방향으로 학습하려는 성질이 있습니다. 데이터가 불균형하면 모델은 다음과 같은 편법을 쓰게 됩니다.

* **다수 클래스 편향:** "그냥 다 '승인'이라고 답하면 90%는 맞겠지?"라고 판단해버립니다.
* **소수 클래스 무시:** 정작 우리가 찾아내야 할 '거절' 고객의 특징을 제대로 배우지 못해 예측력이 떨어집니다.
* **정확도의 함정:** 모델이 아무것도 안 배우고 '전부 승인'이라고만 해도 정확도가 90%가 나오기 때문에, 모델이 잘 작동한다고 착각하게 만듭니다.



---

## 2. 해결 방법 (모델 구축 시 적용할 점)

이 문제를 해결하기 위해 크게 세 가지 관점에서 접근합니다.

### ① 데이터 관점 (Resampling)
* **언더샘플링(Undersampling):** 많은 쪽의 데이터를 줄여서 균형을 맞춥니다. (데이터 손실 위험)
* **오버샘플링(Oversampling):** 적은 쪽의 데이터를 복제하거나 생성합니다. 대표적으로 **SMOTE** 기법이 있으며, 적은 데이터 사이의 특징을 분석해 가상의 데이터를 만들어냅니다.

### ② 모델 관점 (Cost-Sensitive Learning)
* **가중치 부여:** 모델 학습 시 소수 클래스(거절)를 틀렸을 때 더 큰 벌점을 줍니다. 앞서 언급한 `class_weight='balanced'` 옵션이 이 역할을 합니다.

### ③ 평가 지표 관점 (Evaluation Metrics)
* 정확도를 버리고 **정밀도(Precision), 재현율(Recall), F1-Score**를 확인해야 합니다.
    * **재현율(Recall):** "실제로 거절될 사람 중 모델이 몇 명이나 거절로 맞췄는가?" (금융 사고 예방에 중요)
    * **정밀도(Precision):** "거절이라고 예측한 사람 중 실제 거절된 사람은 몇 명인가?" (우량 고객을 놓치지 않는 데 중요)

---

## 3. 요약

클래스 불균형은 ** "시험 문제 100개 중 정답이 'A'인 게 95개인 상황" **과 같습니다. 공부를 안 해도 A만 찍으면 95점을 받지만, 정작 나머지 5개(B, C, D)는 하나도 못 맞히는 우를 범하게 됩니다. 

### 2. 전처리

In [14]:
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler

X = df.drop('승인여부', axis=1)
y = df['승인여부']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # stratify로 비율 유지
)



## 스케일링
> 머신러닝 모델 학습 전, 서로 다른 단위나 범위를 가진 숫자 데이터를 평균이 0, 표준편차가 1이 되도록 변환하는 표준화(Standardization) 과정

- 만약 스케일링을 하지 않으면, 모델(특히 Logistic Regression)은 값이 큰 '연봉'이 '나이'보다 훨씬 중요한 변수라고 착각하게 됩니다. 스케일링을 거치면 모든 변수가 동일한 선상에서 공정하게 비교됩니다.

In [7]:
# Logistic Regression은 스케일링 필요
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. 모델 학습

모델 정의

In [ ]:
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

# class_weight='balanced: 가중치 부여-모델 학습 시 소수 클래스(거절)를 틀렸을 때 더 큰 벌점을 줍니다.
models = {
    'LogisticRegression': LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    'LightGBM': LGBMClassifier(class_weight='balanced', random_state=42, n_estimators=300)
}


lr = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)

lgbm = LGBMClassifier(class_weight='balanced', random_state=42, n_estimators=300)
lgbm.fit(X_train, y_train)  # LightGBM은 스케일링 불필요


[LightGBM] [Info] Number of positive: 2847, number of negative: 1153
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000648 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1109
[LightGBM] [Info] Number of data points in the train set: 4000, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


LGBMClassifier(class_weight='balanced', n_estimators=300, random_state=42)

불균형 비율 계산

In [13]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos = neg_count / pos_count

print(neg_count)
print(pos_count)
print(f'{scale_pos:.2f}')

1153
2847
0.40


In [ ]:
# 5-Fold Stratified CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

cv_results = {}
for name, model in models.items():
    result = cross_validate(
        model, X_train_scaled, y_train, cv=cv, scoring=scoring, n_jobs=-1
    )
    cv_results[name] = {
        'Accuracy': result['test_accuracy'].mean(),
        'Precision': result['test_precision'].mean(),
        'Recall': result['test_recall'].mean(),
        'F1': result['test_f1'].mean(),
        'ROC-AUC': result['test_roc_auc'].mean(),
    }
    print(f'{name}: F1={cv_results[name]["F1"]:.4f}, AUC={cv_results[name]["ROC-AUC"]:.4f}')

cv_df = pd.DataFrame(cv_results).T.round(4)
cv_df

① StratifiedKFold(n_splits=5, ...)K-Fold: 전체 데이터를 5개($K=5$)의 그룹으로 나눕니다. 4개 그룹으로 학습하고 1개 그룹으로 검증하는 과정을 5번 반복합니다.Stratified(층화): 매우 중요한 부분입니다. 데이터를 나눌 때 원본 데이터의 '승인/거절' 비율을 유지하며 나눕니다. (예: 전체 데이터의 거절 비율이 10%라면, 나뉜 5개의 각 조각도 거절 비율을 10%로 유지)shuffle=True: 데이터의 순서를 섞어서 특정 패턴(예: 시간순 정렬)에 의한 왜곡을 방지합니다.② scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']모델을 평가할 '성적표'의 항목들을 정의합니다.불균형 데이터이므로 F1-Score와 ROC-AUC(모델의 분류 결정 임계값에 상관없는 전체적인 성능)를 포함시킨 것이 매우 적절한 선택입니다.③ cross_validate(..., n_jobs=-1)교차 검증 수행: 설정한 5-Fold 방식에 따라 모델을 학습시키고 성능을 측정합니다.n_jobs=-1: 컴퓨터의 모든 CPU 코어를 사용하여 병렬로 연산합니다. 학습 속도를 비약적으로 높여줍니다.④ cv_results[name] = { ... .mean() }5번의 테스트를 거치면 5개의 성적표가 나옵니다. 이 코드에서는 **5개 성적의 평균(mean)**을 내어 해당 모델의 최종 실력으로 기록합니다. 한 번의 우연(운이 좋아 맞춘 경우)을 배제하기 위함입니다.